In [1]:
import pandas as pd

In [35]:
df = pd.read_csv('heart_disease_uci.csv')
cols_do_drop = ['id', 'dataset']
df = df.drop(columns=cols_do_drop)
df['num'] = df['num'] > 0
df['num'] = df['num'].astype(int)

In [3]:
num_cols = ['age', 'trestbps', 'chol', 'fbs', 'exang', 'oldpeak', 'thalch']
cat_cols =['sex', 'cp', 'restecg', 'slope', 'thal']

In [4]:
from sklearn.model_selection import train_test_split

X = df.drop('num', axis=1)
y = df['num']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), 
    ('encoder', OrdinalEncoder(unknown_value=-1, handle_unknown='use_encoded_value'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', SimpleImputer(strategy='median'), num_cols), 
    ('cat', cat_transformer, cat_cols)
], remainder='passthrough')

In [8]:
X_train_prepped = preprocessor.fit_transform(X_train)
X_test_prepped = preprocessor.transform(X_test)

In [32]:
from xgboost import XGBClassifier

clf = XGBClassifier(
    n_estimators = 1844,
    reg_lambda=500,
    learning_rate=0.05,
    gamma=0, 
    early_stopping_rounds=10)

clf.fit(X_train_prepped, y_train, eval_set=[(X_test_prepped, y_test)])
print(clf.best_iteration)
print(clf.best_score)

[0]	validation_0-logloss:0.68469
[1]	validation_0-logloss:0.68218
[2]	validation_0-logloss:0.67970
[3]	validation_0-logloss:0.67725
[4]	validation_0-logloss:0.67483
[5]	validation_0-logloss:0.67244
[6]	validation_0-logloss:0.67007
[7]	validation_0-logloss:0.66774
[8]	validation_0-logloss:0.66543
[9]	validation_0-logloss:0.66315
[10]	validation_0-logloss:0.66090
[11]	validation_0-logloss:0.65868
[12]	validation_0-logloss:0.65648
[13]	validation_0-logloss:0.65431
[14]	validation_0-logloss:0.65216
[15]	validation_0-logloss:0.65004
[16]	validation_0-logloss:0.64801
[17]	validation_0-logloss:0.64600
[18]	validation_0-logloss:0.64401
[19]	validation_0-logloss:0.64205
[20]	validation_0-logloss:0.64012
[21]	validation_0-logloss:0.63814
[22]	validation_0-logloss:0.63625
[23]	validation_0-logloss:0.63438
[24]	validation_0-logloss:0.63253
[25]	validation_0-logloss:0.63070
[26]	validation_0-logloss:0.62890
[27]	validation_0-logloss:0.62711
[28]	validation_0-logloss:0.62535
[29]	validation_0-loglos

In [33]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

clf_pred = clf.predict(X_test_prepped)
confusion = confusion_matrix(y_test, clf_pred)
cr = classification_report(y_test, clf_pred)
acc_score = accuracy_score(y_test, clf_pred)
print(f"Confusion Matrix: {confusion}")
print(f"\n C Report {cr}")
print(f"\nAccuracy Score: {acc_score}")

Confusion Matrix: [[63 19]
 [10 92]]

 C Report               precision    recall  f1-score   support

           0       0.86      0.77      0.81        82
           1       0.83      0.90      0.86       102

    accuracy                           0.84       184
   macro avg       0.85      0.84      0.84       184
weighted avg       0.84      0.84      0.84       184


Accuracy Score: 0.842391304347826


In [34]:
clf_train_score = clf.score(X_train_prepped, y_train)
clf_test_score = clf.score(X_test_prepped, y_test)


print(f'\n Training Score: {clf_train_score:.4f}')
print(f'\n Test Score:{clf_test_score:.4f}')


 Training Score: 0.8927

 Test Score:0.8424


**The point of Day 9 wasn't to beat Day 8. It was to learn a specific tool and understand when it's useful.
Here's what I actually learned and proved through practice:
What early stopping does — instead of guessing n_estimators, I let the model find its own optimal stopping point. It found 1593, not 1000 or 2000. I couldn't have known that number in advance.
What it's actually for — in a real project you often don't have time to run lambda loops like I did on Day 8. Early stopping lets you throw a large n_estimators at the model and say "stop when you peak." It's a shortcut to a reasonable model fast.
What I proved — when I've already done careful regularization tuning like Day 8, early stopping adds little. But if I had skipped Day 8 entirely and just used early stopping on the raw overfit model, it would have helped significantly.
The order matters in real work too — most practitioners use early stopping as a first pass to find a good n_estimators, then tune regularization on top of that. I did it in reverse, which is why the gains were minimal.**